# Create Blood Cancer UK Awards

Blood Cancer UK (formerly Bloodwise) current research portfolio, scraped from the
bloodcancer.org.uk sitemap (~120 project pages). Title, lead researcher,
institution, related conditions (as description), research type (as scheme),
region. **No amounts or dates published** (section 6.7 waiver). provenance
`blood_cancer_uk`, priority 259.

Source parquet: s3://openalex-ingest/awards/blood_cancer_uk/blood_cancer_uk_projects.parquet
Built by scripts/local/blood_cancer_uk_to_s3.py

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.blood_cancer_uk_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/blood_cancer_uk/blood_cancer_uk_projects.parquet`;


In [ ]:
%sql
-- Check row count (should be ~120)
SELECT COUNT(*) as total_projects FROM openalex.awards.blood_cancer_uk_raw;

In [ ]:
%sql
-- Inspect column names
DESCRIBE openalex.awards.blood_cancer_uk_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.blood_cancer_uk_raw LIMIT 5;

## Step 2: Create Blood Cancer UK Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.blood_cancer_uk_awards
USING delta
AS
WITH
the_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320329013  -- Blood Cancer UK
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.conditions as description,
        f.funder_id,
        g.funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.research_type as funder_scheme,
        'blood_cancer_uk' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        CAST(NULL AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United Kingdom' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.blood_cancer_uk_raw g
    CROSS JOIN the_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'blood_cancer_uk' AND priority = 259;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    259 as priority
FROM openalex.awards.blood_cancer_uk_awards;


## Verification Queries

In [ ]:
%sql
-- Check row count (should be ~127)
SELECT COUNT(*) as total_awards FROM openalex.awards.blood_cancer_uk_awards;

In [ ]:
%sql
-- Sample the data
SELECT 
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    end_year
FROM openalex.awards.blood_cancer_uk_awards 
LIMIT 10;

In [ ]:
%sql
-- Check funder distribution (should all be Blood Cancer UK)
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.blood_cancer_uk_awards
GROUP BY funder.display_name
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check program/scheme distribution
SELECT 
    funder_scheme,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000, 2) as funding_million
FROM openalex.awards.blood_cancer_uk_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check year distribution
SELECT 
    start_year,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000, 1) as funding_thousands
FROM openalex.awards.blood_cancer_uk_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 20;

In [ ]:
%sql
-- Check data completeness
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_institution,
    COUNT(lead_investigator.affiliation.ids) as has_ror,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_with_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(SUM(amount)/1000000, 2) as total_funding_million
FROM openalex.awards.blood_cancer_uk_awards;

In [ ]:
%sql
-- Check top institutions
SELECT 
    lead_investigator.affiliation.name as institution,
    COUNT(*) as cnt,
    ROUND(SUM(amount)/1000000, 2) as funding_million
FROM openalex.awards.blood_cancer_uk_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name
ORDER BY cnt DESC
LIMIT 20;

In [ ]:
%sql
-- Verify data was inserted into openalex_awards_raw
SELECT COUNT(*) as blood_cancer_uk_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'blood_cancer_uk' AND priority = 259;